# 12 — Bronze: Portal da Transparência (CEIS e CNEP)

## Purpose
Ingests CEIS (sanctions register) and CNEP (corporate sanctions) into Bronze.
Two separate tables — schemas are identical except `valorMulta` (CNEP only).

## Input
- `data/raw/portal_transparencia/ceis.json` — full historical base
- `data/raw/portal_transparencia/cnep.json` — full historical base

## Output
- `data/bronze/transp_ceis/_year_month={ref}/data.parquet`
- `data/bronze/transp_cnep/_year_month={ref}/data.parquet`
- `{ref}` = dataReferencia value from source (e.g. 17/04/2026 → 2026-04-17)

## Steps
1. Imports and configuration
2. Schema definition
3. Process files
4. Validation
5. Ops registration

## Notes
- Full reload on every run — source is complete historical base
- STRUCTs flattened with underscore: `pessoa.cnpjFormatado` → `pessoa_cnpjFormatado`
- `fundamentacao` is LIST of STRUCTs → serialized as JSON string via to_json()
- Date format: DD/MM/YYYY (Brazilian) — Silver parses with TRY_STRPTIME
- Sentinel "Sem informação" in date fields — Silver treats as NULL
- `valorMulta` present only in CNEP — not in CEIS schema by design
- `_rescued_data`: populated with unexpected key names when drift detected
- ADR-002: all source columns as STRING


## Step 1 — Imports and configuration

In [ ]:
import json
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import get_project_root, to_sql_path
from duckdb_utils import get_connection
from validation import CheckSuite
from bootstrap_log import append_log, make_entry
from pipeline import check_landing_gate

PROJECT_ROOT = get_project_root()

TRANSP_ROOT = PROJECT_ROOT / "data" / "raw"    / "portal_transparencia"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
DRIFT_LOG   = PROJECT_ROOT / "db"  / "schema_drift_log.json"
LOG_PATH    = PROJECT_ROOT / "db"  / "bootstrap_log.json"

CEIS_PATH = to_sql_path(TRANSP_ROOT / "ceis.json")
CNEP_PATH = to_sql_path(TRANSP_ROOT / "cnep.json")

check_landing_gate(PROJECT_ROOT)

SOURCE_ID  = "portal_transparencia"
BATCH_ID   = str(uuid.uuid4())
STARTED_AT = datetime.now(timezone.utc).isoformat()

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"TRANSP_ROOT  : {TRANSP_ROOT}")
print(f"BATCH_ID     : {BATCH_ID}")
print(f"STARTED_AT   : {STARTED_AT}")


## Step 2 — Schema definition

In [ ]:
# ---------------------------------------------------------------------------
# Shared scalar columns — present in both CEIS and CNEP
# ---------------------------------------------------------------------------
SHARED_SCALAR_COLS = [
    "id",
    "dataReferencia",
    "dataInicioSancao",
    "dataFimSancao",
    "dataPublicacaoSancao",
    "dataTransitadoJulgado",
    "dataOrigemInformacao",
    "textoPublicacao",
    "linkPublicacao",
    "detalhamentoPublicacao",
    "numeroProcesso",
    "abrangenciaDefinidaDecisaoJudicial",
    "informacoesAdicionaisDoOrgaoSancionador",
]

# Flattened STRUCT columns — shared by CEIS and CNEP
SHARED_STRUCT_COLS = [
    "tipoSancao_descricaoResumida",
    "tipoSancao_descricaoPortal",
    "fonteSancao_nomeExibicao",
    "fonteSancao_telefoneContato",
    "fonteSancao_enderecoContato",
    "orgaoSancionador_nome",
    "orgaoSancionador_siglaUf",
    "orgaoSancionador_poder",
    "orgaoSancionador_esfera",
    "sancionado_nome",
    "sancionado_codigoFormatado",
    "pessoa_id",
    "pessoa_cpfFormatado",
    "pessoa_cnpjFormatado",
    "pessoa_numeroInscricaoSocial",
    "pessoa_nome",
    "pessoa_razaoSocialReceita",
    "pessoa_nomeFantasiaReceita",
    "pessoa_tipo",
    "fundamentacao",   # LIST of STRUCTs → JSON string via to_json()
]

# CNEP-only column
CNEP_ONLY_COLS = ["valorMulta"]

CEIS_COLS     = SHARED_SCALAR_COLS + SHARED_STRUCT_COLS
CNEP_COLS     = SHARED_SCALAR_COLS + SHARED_STRUCT_COLS + CNEP_ONLY_COLS
TECHNICAL_COLS = [
    "_rescued_data",   # unexpected top-level key names when drift detected
    "_source_id",
    "_batch_id",
    "_ingested_at",
    "_source_file",
    "_year_month",
]

# STRUCT_MAP — used in _build_select() for dot-access flattening
STRUCT_MAP = {
    "tipoSancao"      : ["descricaoResumida", "descricaoPortal"],
    "fonteSancao"     : ["nomeExibicao", "telefoneContato", "enderecoContato"],
    "orgaoSancionador": ["nome", "siglaUf", "poder", "esfera"],
    "sancionado"      : ["nome", "codigoFormatado"],
    "pessoa"          : ["id", "cpfFormatado", "cnpjFormatado", "numeroInscricaoSocial",
                         "nome", "razaoSocialReceita", "nomeFantasiaReceita", "tipo"],
}

# Known top-level keys — drift = any key not in these sets
KNOWN_ROOTS_SHARED = set(SHARED_SCALAR_COLS) | set(STRUCT_MAP.keys()) | {"fundamentacao"}
KNOWN_ROOTS_CEIS   = KNOWN_ROOTS_SHARED
KNOWN_ROOTS_CNEP   = KNOWN_ROOTS_SHARED | {"valorMulta"}

print(f"CEIS columns   : {len(CEIS_COLS)} source + {len(TECHNICAL_COLS)} technical = {len(CEIS_COLS)+len(TECHNICAL_COLS)} total")
print(f"CNEP columns   : {len(CNEP_COLS)} source + {len(TECHNICAL_COLS)} technical = {len(CNEP_COLS)+len(TECHNICAL_COLS)} total")
print(f"CNEP-only      : {CNEP_ONLY_COLS}")
print(f"fundamentacao  : LIST of STRUCTs -> JSON string via to_json()")


## Step 3 — Process files

In [ ]:
def _build_select(scalar_cols: list, include_valor_multa: bool = False) -> str:
    """
    Build the SELECT SQL for Bronze transformation.

    - Scalar columns: CAST(col AS VARCHAR)
    - STRUCT subfields: CAST(struct."field" AS VARCHAR) AS struct_field
    - fundamentacao: CAST(to_json("fundamentacao") AS VARCHAR)
    - valorMulta: CAST("valorMulta" AS VARCHAR) — CNEP only

    Parameters
    ----------
    scalar_cols         : list of top-level scalar column names
    include_valor_multa : True for CNEP, False for CEIS
    """
    parts = []

    for col in scalar_cols:
        parts.append(f'CAST("{col}" AS VARCHAR) AS "{col}"')

    for struct_root, fields in STRUCT_MAP.items():
        for field in fields:
            col_name = f"{struct_root}_{field}"
            parts.append(
                f'CAST("{struct_root}"."{field}" AS VARCHAR) AS "{col_name}"'
            )

    # fundamentacao — LIST of STRUCTs serialized to JSON string
    parts.append('CAST(to_json("fundamentacao") AS VARCHAR) AS "fundamentacao"')

    if include_valor_multa:
        parts.append('CAST("valorMulta" AS VARCHAR) AS "valorMulta"')

    return ",\n    ".join(parts)


def _log_drift(source_name: str, unexpected_keys: set) -> None:
    """
    Append a schema drift event to schema_drift_log.json.

    Parameters
    ----------
    source_name    : "ceis" or "cnep"
    unexpected_keys: set of top-level keys not in the known schema
    """
    event = {
        "batch_id"       : BATCH_ID,
        "source_id"      : SOURCE_ID,
        "object"         : f"bronze_transp_{source_name}",
        "event_type"     : "SCHEMA_DRIFT",
        "severity"       : "WARN",
        "action"         : "CONTINUE",
        "unexpected_keys": sorted(unexpected_keys),
        "logged_at"      : datetime.now(timezone.utc).isoformat(),
    }
    existing = []
    if DRIFT_LOG.exists():
        with open(DRIFT_LOG, "r", encoding="utf-8") as f:
            try:
                existing = json.load(f)
            except json.JSONDecodeError:
                existing = []
    existing.append(event)
    DRIFT_LOG.parent.mkdir(parents=True, exist_ok=True)
    with open(DRIFT_LOG, "w", encoding="utf-8") as f:
        json.dump(existing, f, ensure_ascii=False, indent=2)


def process_source(
    source_name: str,
    source_path: str,
    scalar_cols: list,
    known_roots: set,
    include_valor_multa: bool,
    con,
) -> dict:
    """
    Process one source file (CEIS or CNEP) into a Bronze Parquet partition.

    Full reload: partition is overwritten on every run.

    _rescued_data strategy for JSON sources:
    - Slow path (per-record) is impractical for 22k+ records with complex STRUCTs
    - When drift is detected, _rescued_data = JSON list of unexpected key names
    - The actual unexpected values are preserved in the source JSON file

    Parameters
    ----------
    source_name         : "ceis" or "cnep"
    source_path         : forward-slash path to the JSON file
    scalar_cols         : expected scalar column names
    known_roots         : set of expected top-level keys
    include_valor_multa : True for CNEP
    con                 : DuckDB connection
    """
    # Derive _year_month from dataReferencia in source: DD/MM/YYYY → YYYY-MM-DD
    ref_row = con.execute(
        f"SELECT dataReferencia FROM read_json_auto('{source_path}') LIMIT 1"
    ).fetchone()
    if ref_row and ref_row[0]:
        parts     = ref_row[0].strip().split("/")
        year_month = f"{parts[2]}-{parts[1]}-{parts[0]}"
    else:
        year_month = datetime.now(timezone.utc).strftime("%Y-%m-%d")

    table_name    = f"transp_{source_name}"
    partition_dir = BRONZE_ROOT / table_name / f"_year_month={year_month}"
    partition_dir.mkdir(parents=True, exist_ok=True)
    output_file  = partition_dir / "data.parquet"
    output_path  = to_sql_path(output_file)

    try:
        # Schema drift detection
        source_cols = {row[0] for row in con.execute(
            f"DESCRIBE SELECT * FROM read_json_auto('{source_path}') LIMIT 0"
        ).fetchall()}
        unexpected = source_cols - known_roots
        has_drift  = bool(unexpected)

        if has_drift:
            _log_drift(source_name, unexpected)
            print(f"  [DRIFT] {source_name}: unexpected keys {sorted(unexpected)}")

        # _rescued_data: JSON list of unexpected key names when drift detected
        # Full per-record slow path is impractical for complex nested JSON
        rescued_expr = (
            f"'{json.dumps(sorted(unexpected))}'::VARCHAR"
            if has_drift else
            "NULL::VARCHAR"
        )

        select_sql = _build_select(scalar_cols, include_valor_multa)

        con.execute(f"""
            COPY (
                SELECT
                    {select_sql},
                    {rescued_expr}     AS _rescued_data,
                    '{SOURCE_ID}'     AS _source_id,
                    '{BATCH_ID}'      AS _batch_id,
                    CURRENT_TIMESTAMP  AS _ingested_at,
                    '{source_path}'   AS _source_file,
                    '{year_month}'    AS _year_month
                FROM read_json_auto('{source_path}')
            ) TO '{output_path}'
            (FORMAT PARQUET, COMPRESSION SNAPPY)
        """)

        records = con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{output_path}')"
        ).fetchone()[0]

        return {
            "source"      : source_name,
            "status"      : "SUCCESS",
            "records"     : records,
            "year_month"  : year_month,
            "output_file" : output_file,
            "output_path" : output_path,
            "has_drift"   : has_drift,
            "error"       : None,
        }

    except Exception as e:
        return {
            "source"      : source_name,
            "status"      : "ERROR",
            "records"     : 0,
            "year_month"  : year_month,
            "output_file" : output_file,
            "output_path" : to_sql_path(output_file),
            "has_drift"   : False,
            "error"       : str(e),
        }


# ---------------------------------------------------------------------------
# Execute — process CEIS then CNEP
# ---------------------------------------------------------------------------
results = []
con = get_connection()  # in-memory — Parquet I/O does not require persistent DB

for source_name, source_path, scalar_cols, known_roots, include_multa in [
    ("ceis", CEIS_PATH, SHARED_SCALAR_COLS, KNOWN_ROOTS_CEIS, False),
    ("cnep", CNEP_PATH, SHARED_SCALAR_COLS, KNOWN_ROOTS_CNEP, True),
]:
    print(f"Processing {source_name.upper()}...")
    res = process_source(source_name, source_path, scalar_cols,
                         known_roots, include_multa, con)
    results.append(res)
    icon         = "OK " if res["status"] == "SUCCESS" else "ERR"
    drift_suffix = " [DRIFT]" if res["has_drift"] else ""
    print(f"  [{icon}] {res['source'].upper():<6} {res['status']:<8}"
          f" {res['records']:>8,} records"
          f" | _year_month={res['year_month']}{drift_suffix}")

con.close()

error_count   = sum(1 for r in results if r["status"] == "ERROR")
total_records = sum(r["records"] for r in results)
has_drift_any = any(r["has_drift"] for r in results)

print()
print(f"Total records : {total_records:,}")
print(f"Errors        : {error_count}")
print(f"Schema drift  : {has_drift_any}")


## Step 4 — Validation

In [ ]:
suite   = CheckSuite("bronze_transp_ceis_cnep")
con_val = get_connection()

ceis_r = next((r for r in results if r["source"] == "ceis"), None)
cnep_r = next((r for r in results if r["source"] == "cnep"), None)

# Check 1 — no processing errors
suite.add("No processing errors", error_count == 0, f"{error_count} error(s)")

# Check 2 — both Parquet files exist
both_exist = all(
    r["output_file"].exists()
    for r in results if r["status"] == "SUCCESS"
)
suite.add("Both Parquet files exist", both_exist, "")

# Check 3 — CEIS has records
if ceis_r and ceis_r["status"] == "SUCCESS":
    suite.add("CEIS records > 0", ceis_r["records"] > 0, f"{ceis_r['records']:,}")

# Check 4 — CNEP has records
if cnep_r and cnep_r["status"] == "SUCCESS":
    suite.add("CNEP records > 0", cnep_r["records"] > 0, f"{cnep_r['records']:,}")

# Check 5 — CEIS column count
if ceis_r and ceis_r["status"] == "SUCCESS":
    actual = con_val.execute(
        f"SELECT COUNT(*) FROM (DESCRIBE SELECT * FROM read_parquet('{ceis_r['output_path']}') LIMIT 0)"
    ).fetchone()[0]
    expected = len(CEIS_COLS) + len(TECHNICAL_COLS)
    suite.add("CEIS column count correct", actual == expected, f"found={actual} expected={expected}")

# Check 6 — CNEP column count
if cnep_r and cnep_r["status"] == "SUCCESS":
    actual = con_val.execute(
        f"SELECT COUNT(*) FROM (DESCRIBE SELECT * FROM read_parquet('{cnep_r['output_path']}') LIMIT 0)"
    ).fetchone()[0]
    expected = len(CNEP_COLS) + len(TECHNICAL_COLS)
    suite.add("CNEP column count correct", actual == expected, f"found={actual} expected={expected}")

# Check 7 — valorMulta present in CNEP, absent in CEIS
if cnep_r and cnep_r["status"] == "SUCCESS":
    cnep_cols = {row[0] for row in con_val.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{cnep_r['output_path']}') LIMIT 0"
    ).fetchall()}
    suite.add("valorMulta present in CNEP", "valorMulta" in cnep_cols, "")

if ceis_r and ceis_r["status"] == "SUCCESS":
    ceis_cols = {row[0] for row in con_val.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{ceis_r['output_path']}') LIMIT 0"
    ).fetchall()}
    suite.add("valorMulta absent from CEIS", "valorMulta" not in ceis_cols, "")

# Check 8 — sentinel "Sem informação" preserved as STRING in CNEP
if cnep_r and cnep_r["status"] == "SUCCESS":
    sentinel_count = con_val.execute(f"""
        SELECT COUNT(*) FROM read_parquet('{cnep_r["output_path"]}')
        WHERE dataFimSancao = 'Sem informa\u00e7\u00e3o'
    """).fetchone()[0]
    suite.add(
        "Sentinel Sem informacao preserved in CNEP",
        sentinel_count > 0,
        f"{sentinel_count:,} records (expected ~97% of CNEP)",
    )

# Check 9 — schema drift monitored (always PASS)
suite.add(
    "Schema drift monitored",
    True,
    "drift detected — check schema_drift_log.json" if has_drift_any else "none detected",
)

con_val.close()  # close before potential raise
suite.report()
suite.assert_all_pass()


## Step 5 — Ops registration

In [ ]:
files_written = sum(1 for r in results if r["status"] == "SUCCESS")
bytes_written = sum(
    r["output_file"].stat().st_size
    for r in results if r["status"] == "SUCCESS" and r["output_file"].exists()
)
errors_detail = (
    "; ".join(f"{r['source']}: {r['error'][:60]}" for r in results if r["status"] == "ERROR")
    if error_count > 0 else None
)
# period: reference date from first successful result
period = next((r["year_month"] for r in results if r["status"] == "SUCCESS"), "unknown")

entry = make_entry(
    source_id     = SOURCE_ID,
    period        = period,
    status        = "SUCCESS" if error_count == 0 else "PARTIAL",
    records       = total_records,
    bytes_written = bytes_written,
    started_at    = STARTED_AT,
    finished_at   = datetime.now(timezone.utc).isoformat(),
    error_message = errors_detail,
    batch_id         = BATCH_ID,
    layer            = "bronze",
    object           = "transp_ceis_cnep",
    files            = files_written,
    has_rescued_data = has_drift_any,
    drift_months     = 1 if has_drift_any else 0,
)

append_log(LOG_PATH, entry)

print(f"Batch registered   : {BATCH_ID}")
print(f"Status             : {entry['status']}")
print(f"Records            : {total_records:,}")
print(f"Bytes written      : {bytes_written / 1024:.1f} KB")
print(f"Has rescued data   : {has_drift_any}")
print(f"Period             : {period}")
print(f"Log                : {LOG_PATH}")
